# NHL Home Game Predictor — EDA

Explore data distributions, win/loss patterns, per-team trends, and correlations.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pipeline.data_loader import load_all_seasons

sns.set_theme(style='whitegrid')
df = load_all_seasons()
print(df.shape)
df.head()

In [ ]:
# Missing values
df.isnull().sum()[df.isnull().sum() > 0]

In [ ]:
# Win rate baseline
home_win_rate = df['result'].eq('W').mean()
print(f'Home win rate: {home_win_rate:.3f}')

In [ ]:
# Win/loss distribution per season
df.groupby(['season', 'result']).size().unstack().plot(kind='bar', title='W/L per Season')
plt.tight_layout()
plt.show()

In [ ]:
# Numeric feature distributions
numeric_cols = df.select_dtypes(include='number').columns.tolist()
df[numeric_cols].hist(bins=30, figsize=(20, 14))
plt.suptitle('Feature distributions')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix
corr = df[numeric_cols].corr()
plt.figure(figsize=(16, 12))
sns.heatmap(corr, cmap='RdBu_r', center=0, fmt='.2f', annot=False)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# PDO distribution — luck indicator
df['win_binary'] = df['result'].eq('W').astype(int)
plt.figure(figsize=(8, 4))
df.groupby('win_binary')['pdo'].plot.kde()
plt.axvline(100, color='k', linestyle='--', label='PDO=100')
plt.legend(['Loss', 'Win', 'Neutral'])
plt.title('PDO Distribution by Outcome')
plt.show()

In [ ]:
# Per-team win rates
team_win_rates = df.groupby('team')['win_binary'].mean().sort_values(ascending=False)
team_win_rates.plot(kind='bar', figsize=(14, 5), title='Home Win Rate by Team')
plt.axhline(home_win_rate, color='r', linestyle='--', label='Average')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Key stats by win/loss
key_stats = ['goals_for', 'goals_against', 'team_sog', 'opp_sog', 'corsi_pct', 'fenwick_pct', 'pdo']
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flat, key_stats):
    df.boxplot(column=col, by='result', ax=ax)
    ax.set_title(col)
    ax.set_xlabel('')
plt.suptitle('Key Stats by Win/Loss')
plt.tight_layout()
plt.show()